In [217]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

import pickle

In [218]:
df=pd.read_csv('Churn_Modelling.csv',)
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [219]:
df=df.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [220]:
label_encode_gender=LabelEncoder()
df['Gender']=label_encode_gender.fit_transform(df['Gender'])
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [221]:
##onehotencoder
from sklearn.preprocessing import OneHotEncoder
One_hot=OneHotEncoder()
geo_encode=One_hot.fit_transform(df[['Geography']])
geo_encode.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [222]:
geo_encode_df=pd.DataFrame(geo_encode.toarray(),columns=One_hot.get_feature_names_out(['Geography'])
 )
geo_encode_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [223]:
df= pd.concat([df.drop('Geography' , axis=1),geo_encode_df],axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [224]:
with open('label_encode_gender.pkl' ,'wb') as file:
   pickle.dump(label_encode_gender,file) 

with open('geo_encode.pkl' ,'wb') as file:
   pickle.dump(One_hot,file) 

In [225]:
## now separating dependent and independent variables

X=df.drop(['EstimatedSalary'], axis =1) 
Y=df['EstimatedSalary']

In [226]:
X_train , X_test , Y_train , Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

In [227]:
scalar=StandardScaler()
X_train= scalar.fit_transform(X_train)
X_test=scalar.transform(X_test)

In [228]:
with open("scalar.pkl","wb") as file:
  pickle.dump(scalar,file)

In [229]:
import tensorflow as tf 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard
import datetime 




In [230]:
model=Sequential([
 Dense(64,activation='relu',input_shape=(X_train.shape[1],)),   ##hideen layer 1
 Dense(32,activation='relu'),##hidden layer 2
 Dense(1) ## output layer 

])

In [231]:
model.summary()

Model: "sequential_8"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_24 (Dense)            (None, 64)                832       
                                                                 
 dense_25 (Dense)            (None, 32)                2080      
                                                                 
 dense_26 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [232]:

model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [233]:
##lets create log table 
logs_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_callback=TensorBoard(log_dir=logs_dir, histogram_freq=1)

In [234]:
Early_Stopping=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [235]:
history=model.fit(
  X_train,Y_train,
  validation_data=(X_test,Y_test),
  epochs=100,
  callbacks=[tensor_callback,Early_Stopping]
)


Epoch 1/100
250/250 [==============================] - 2s 4ms/step - loss: 100373.6484 - mae: 100373.6484 - val_loss: 98502.4609 - val_mae: 98502.4609
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99583.4609 - mae: 99583.4609 - val_loss: 96914.6250 - val_mae: 96914.6250
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 96826.8594 - mae: 96826.8594 - val_loss: 92870.6016 - val_mae: 92870.6016
Epoch 4/100
250/250 [==============================] - 1s 5ms/step - loss: 91451.3984 - mae: 91451.3984 - val_loss: 86183.4453 - val_mae: 86183.4453
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 83540.0312 - mae: 83540.0312 - val_loss: 77394.8672 - val_mae: 77394.8672
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 74242.5469 - mae: 74242.5469 - val_loss: 68361.8984 - val_mae: 68361.8984
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 65309.7969 - mae: 65309.7969 

In [236]:
model.save('model.h6')

INFO:tensorflow:Assets written to: model.h6\assets


INFO:tensorflow:Assets written to: model.h6\assets


In [237]:
import numpy as np 
from tensorflow.keras.models import load_model


In [238]:
model = load_model('model.h6')